# Construction des références Att — étape 2 (batch, tous les PU)

Ce notebook exécute l'**étape 2** du flux en 2 temps décrit dans
`documents/confidentiel/OPERATION.md` :

1. analyse du document opératoire `OPxxxxVA.doc` → `OPxxxx_pas_reference.csv`
   (manuel/assisté, une fois par PU — **pas** ce notebook) ;
2. **chargement des données réelles + reconstruction des opérations +
   export d'un bundle de référence JSON par opération** — ce notebook.

C'est la version **interactive** (sortie détaillée, tag par tag, pour
vérifier visuellement le résultat avant de faire confiance à la référence
produite) du job batch `build_att_references.py` (sans supervision, pensé
pour un cron/scheduler). **Les deux réutilisent exactement la même fonction**
(`build_att_references.process_tag`) : aucune logique dupliquée, seule la
présentation diffère.

**Pré-requis** — chaque tag de `att_tags_config.py` doit avoir sa table
`pas_reference` déjà construite et validée (étape 1, cf. `OPERATION.md`) ;
les tags sans cette table sont ignorés (avertissement affiché, pas d'erreur
bloquante).

**Ce que ce notebook ne fait pas** — pas d'exploration ni de graphique de
diagnostic (distribution des durées, matrice de transition, carte SPC...) :
pour ça, voir `OI_ATT_v0.ipynb`, l'analyse interactive détaillée d'un tag à
la fois. Ce notebook-ci est volontairement minimal : charger, reconstruire,
exporter, résumer — pour tous les tags d'un coup.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from tools.OI_class_OP import OI_DataProcessor
import att_analysis as atta
from att_tags_config import TAGS, has_pas_reference
from build_att_references import process_tag, CHAINING_WINDOW, OUTPUT_DIR, URL_BASE, START, END

pd.set_option('display.max_columns', None)

## 1. Chargement des données — tous les tags en un seul appel

Même `OI_DataProcessor` que dans `OI_ATT_v0.ipynb`, mais ici sur **tous**
les tags de `att_tags_config.TAGS` d'un coup (un seul appel réseau groupé
par `agg`, cf. `OI_DataProcessor.merge()`) — pas de boucle par tag à ce
stade, `processor.data` contient une colonne par tag (`nom`) une fois
`merge()` exécuté.

In [2]:
processor = OI_DataProcessor(
    url_base=URL_BASE,
    start=START,
    end=END,
    tags_selected=TAGS,
    tags_other=[],
    interval='PT01M',
    verbose=False,
    agg='FIRST',
)

processor.merge()
processor.data.describe()

,PU1410VA,PU1420VA,PU1430VA,PU1510VA,PU1520VA,PU1530VA,PU1540VA,PU1610VA,PU2810VA,PU2910VA,PU2310VA,PU2320VA,PU2340VA,PU2330VA,PU2410VA,PU2411VA,PU2420VA,PU2510VA,PU2520VA,PU3110VA,PU3120VA,PU3130VA,PU3210VA,PU3220VA,PU3230VA,PU3310VA,PU3320VA,PU3330VA
count,828348.000000,524326.000000,823966.000000,534898.000000,541094.000000,534483.000000,553837.000000,559266.000000,513513.000000,512010.000000,526742.000000,533640.000000,523881.000000,512622.000000,528089.000000,523435.000000,535270.000000,511728.000000,512367.000000,537428.000000,577865.000000,514506.000000,545108.000000,541542.000000,524872.000000,530952.000000,565857.000000,528092.000000
mean,1189.774942,446.104357,1176.887426,531.190264,529.306304,484.582690,632.510847,953.441788,1193.146133,767.017890,731.791987,1107.538378,450.895795,619.042745,819.905859,598.924250,803.786603,785.645851,592.401901,1212.341908,1034.050664,599.179077,1016.149123,683.141908,485.302386,915.351284,893.837072,870.262278
std,662.684805,297.575772,785.005670,390.563093,347.758669,272.135597,483.489801,517.632576,401.083986,354.233619,480.205330,742.389106,150.026193,255.874191,475.028028,229.516902,480.398441,331.728022,249.940457,800.751754,783.153211,146.625877,592.616315,358.129471,231.020461,664.612128,601.739881,310.603160
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,730.000000,230.000000,110.000000,0.000000,340.000000,320.000000,310.000000,540.000000,1020.000000,850.000000,240.000000,420.000000,410.000000,670.000000,470.000000,520.000000,510.000000,950.000000,720.000000,218.000000,270.000000,620.000000,540.000000,510.000000,430.000000,410.000000,410.000000,930.000000
50%,1410.000000,230.000000,1720.000000,820.000000,510.000000,610.000000,610.000000,1250.000000,1410.000000,960.000000,720.000000,750.000000,410.000000,750.000000,1110.000000,720.000000,810.000000,950.000000,720.000000,1530.000000,1050.000000,620.000000,1150.000000,630.000000,440.000000,715.000000,1070.000000,930.000000
75%,1650.000000,740.000000,1720.000000,820.000000,810.000000,710.000000,1020.000000,1250.000000,1410.000000,960.000000,1220.000000,1950.000000,430.000000,750.000000,1250.000000,720.000000,1350.000000,950.000000,720.000000,2010.000000,1870.000000,620.000000,1620.000000,1030.000000,440.000000,1320.000000,1220.000000,930.000000
max,2190.000000,1460.000000,2740.000000,1050.000000,1230.000000,860.000000,1630.000000,1710.000000,1580.000000,1380.000000,1660.000000,2180.000000,740.000000,930.000000,1390.000000,750.000000,1450.000000,1330.000000,830.000000,2090.000000,2190.000000,720.000000,1760.000000,1420.000000,980.000000,2320.000000,2260.000000,1390.000000


## 2. Reconstruction + export de la référence, tag par tag

Pour chaque tag : nettoyage (`att_analysis.clean_dataframe`, exclusion
d'Att=0), extraction des pas, étiquetage par la table process
(`pas_reference`), reconstruction des opérations (défauts inclus), temps de
cycle/chaînage, puis assemblage du bundle complet
(`att_analysis.build_reference` — statistiques globales, détail par pas,
Pareto des défauts, **enchaînements inter-opérations**). Toute cette
séquence est encapsulée dans `build_att_references.process_tag`, appelée
ici telle quelle (pas de logique métier dans ce notebook).

Un tag est **ignoré** (avertissement, pas d'erreur) s'il n'a pas de donnée
chargée ou pas de `pas_reference` configurée. Un tag en **échec** (pipeline
qui lève une exception) n'interrompt pas les suivants — l'erreur est
affichée et le batch continue.

In [3]:
references = {}
summary_rows = []

for tag_def in TAGS:
    label, nom = tag_def['tag'], tag_def['nom']

    if nom not in processor.data.columns:
        display(Markdown(f"⚠️ **{label}** : donnée absente de `processor.data`, ignoré."))
        continue
    if not tag_def.get('pas_reference'):
        display(Markdown(f"⚠️ **{label}** : pas de `pas_reference` configurée (étape 1 non faite), ignoré."))
        continue

    try:
        reference = process_tag(tag_def, processor.data)
    except Exception as exc:
        display(Markdown(f"❌ **{label}** : échec du pipeline — `{exc}`"))
        continue

    references[label] = reference

    out_path = Path(OUTPUT_DIR) / f"{label}_reference.json"
    atta.save_reference(reference, str(out_path))

    summary_rows.append({
        'Tag': label,
        'N opérations': reference['n_operations'],
        'Durée totale — médiane (min)': round(reference['duration_min']['median'], 0),
        'Rework — % du temps total': round(reference['rework_duration_min']['pct_of_total_time'], 1),
        '% opérations avec défaut': round(reference['defauts']['pct_operations_with_defaut'], 1),
        'N enchaînements inter-opérations': len(reference['enchainements']),
        'Référence exportée': str(out_path),
    })

    display(Markdown(f"✅ **{label}** — {reference['n_operations']:,} opérations → `{out_path}`"))

summary_table = pd.DataFrame(summary_rows)
display(Markdown("### Synthèse du batch"))
summary_table

✅ **PU1410VA_Att** — 1,767 opérations → `documents/confidentiel/PU1410VA_Att_reference.json`

✅ **PU1420VA_Att** — 2,374 opérations → `documents/confidentiel/PU1420VA_Att_reference.json`

✅ **PU1430VA_Att** — 783 opérations → `documents/confidentiel/PU1430VA_Att_reference.json`

✅ **PU1510VA_Att** — 174 opérations → `documents/confidentiel/PU1510VA_Att_reference.json`

✅ **PU1520VA_Att** — 26 opérations → `documents/confidentiel/PU1520VA_Att_reference.json`

...

✅ **PU3310VA_Att** — 795 opérations → `documents/confidentiel/PU3310VA_Att_reference.json`

✅ **PU3320VA_Att** — 489 opérations → `documents/confidentiel/PU3320VA_Att_reference.json`

✅ **PU3330VA_Att** — 117 opérations → `documents/confidentiel/PU3330VA_Att_reference.json`

### Synthèse du batch

,Tag,N opérations,Durée totale — médiane (min),Rework — % du temps total,% opérations avec défaut,N enchaînements inter-opérations,Référence exportée
0,PU1410VA_Att,1767,607.0,51.8,37.1,2,documents/confidentiel/PU1410VA_Att_reference....
1,PU1420VA_Att,2374,547.0,36.3,0.8,3,documents/confidentiel/PU1420VA_Att_reference....
2,PU1430VA_Att,783,1769.0,38.8,47.4,0,documents/confidentiel/PU1430VA_Att_reference....
3,PU1510VA_Att,174,5371.0,87.5,90.2,1,documents/confidentiel/PU1510VA_Att_reference....
4,PU1520VA_Att,26,53264.0,97.5,84.6,2,documents/confidentiel/PU1520VA_Att_reference....
5,PU1530VA_Att,39,25927.0,98.3,87.2,2,documents/confidentiel/PU1530VA_Att_reference....
6,PU1540VA_Att,55,26089.0,95.0,85.5,1,documents/confidentiel/PU1540VA_Att_reference....
7,PU1610VA_Att,325,3042.0,78.1,81.8,0,documents/confidentiel/PU1610VA_Att_reference....
8,PU2810VA_Att,76,5064.0,83.7,50.0,0,documents/confidentiel/PU2810VA_Att_reference....
9,PU2910VA_Att,464,2002.0,33.4,39.0,1,documents/confidentiel/PU2910VA_Att_reference....


## 3. Enchaînements inter-opérations détectés (rappel)

Vue consolidée des dépendances inter-opérations (`operation_liee`,
`sens_liaison`) de chaque tag traité — identifiées à l'étape 1 (analyse du
document), simplement reprises ici depuis le bundle de référence pour
vérification rapide sans rouvrir le CSV. Une table vide signifie qu'aucun
enchaînement n'a été documenté pour ce tag dans sa `pas_reference` (soit il
n'y en a réellement pas, soit l'étape 1 reste à compléter — cf.
`OPERATION.md`, section 4bis).

In [4]:
for label, reference in references.items():
    display(Markdown(f"#### {label}"))
    if reference['enchainements']:
        display(pd.DataFrame(reference['enchainements']))
    else:
        display(Markdown("_Aucun enchaînement inter-opérations documenté pour ce tag._"))

#### PU1410VA_Att

,pas_num,code_court,operation_liee,sens_liaison,detail
0,20,REFROIDISS.,OP1610VA,BIDIRECTIONNEL,"Refroidissement du reacteur, puis transfert co..."
1,21,EGOUTTAGE,OP1420VA,BIDIRECTIONNEL,"""La vidange du reacteur K14000 est pilotee par..."


#### PU1420VA_Att

,pas_num,code_court,operation_liee,sens_liaison,detail
0,2,INIT.OPER.,OP1410VA,ATTEND,Verification de niveaux avant demarrage (LX089...
1,3,NEUTRAL.1,OP1410VA,ATTEND,Attente de la fin de vidange de OP1410VA (Acet...
2,4,CH.MELANGE,OP1410VA,ATTEND,Charge du melange reactionnel (XV14406VA ouver...


#### PU1430VA_Att

...

,pas_num,code_court,operation_liee,sens_liaison,detail
0,4,ATT.OP3230VA,OP3230VA,BIDIRECTIONNEL,"Attente ""OP3230VA prete a etre vidangee"" puis ..."
1,23,VIDANGE,OP3320VA,BIDIRECTIONNEL,"""DIALOGUE AVEC OP3320VA"" : attente ""Reponse de..."


#### PU3320VA_Att

,pas_num,code_court,operation_liee,sens_liaison,detail
0,4,TRANSF.33000,OP3310VA,ATTEND,"Attente ""Autorisation de vidange de KA33000"" (..."
1,6,TRANSF.RINC.,OP3310VA,ATTEND,"Transfert du rincage (vanne XV33414VA ouverte,..."
2,9,LAVAGE SODE,OP3321VA,ATTEND,"Attente ""OP3321VA hors Pas de defaut"" avant de..."
3,10,CH.SOUD.MET,OP3321VA,BIDIRECTIONNEL,Charge (niveau LI33201VA<L) avec branchement s...
4,12,CH.EAU SODEE,OP3321VA,BIDIRECTIONNEL,Charge (avec branchement Acetate/Propionate) p...
5,14,LAVAGE SALIN,OP3321VA,ATTEND,"Attente ""OP3321VA hors Pas de defaut"" avant de..."
6,15,CH.L.SALIN 1,OP3321VA,BIDIRECTIONNEL,"Charge (niveau LI33202VA<L, timer K3345, vanne..."
7,17,CH.L.SALIN 2,OP3321VA,BIDIRECTIONNEL,"Dialogue avec OP3321VA : attente ""Autorisation..."
8,20,VID.ROUTINE,OP3330VA,BIDIRECTIONNEL,"Attente ""Autorisation OP3330VA"" puis vidange (..."
9,22,RINC.DERN.OPE,OP3330VA,ENVOIE,Rincage (vannes XV33431VA/XV33409VA/XV33424VA ...


#### PU3330VA_Att

,pas_num,code_court,operation_liee,sens_liaison,detail
0,7,ALIMENTATION,OP3320VA,ATTEND,"Alimentation (timer, niveau LX33918VA, vannes ..."
1,8,ALIM.3ETAGE,OP3320VA,ATTEND,"Alimentation 3eme etage, avec le meme controle..."
2,9,M.STABILISEE,OP3320VA,ATTEND,"Marche stabilisee ; peut etre interrompue ""Si ..."
3,13,ARRET TOTAL,OP3320VA,ENVOIE,Retour a la pression atmospherique (timer K336...


## Pour aller plus loin

- **Ajouter un nouveau PU** : construire et valider son
  `OPxxxx_pas_reference.csv` (cf. `OPERATION.md`), l'ajouter à
  `att_tags_config.TAGS`, puis relancer ce notebook (ou
  `build_att_references.py`) — rien d'autre à modifier.
- **Automatiser** : `python build_att_references.py` fait exactement ce que
  fait ce notebook (sections 1 et 2), sans sortie interactive — à
  programmer (cron/scheduler) pour tenir les références à jour au fil de
  l'eau.
- **Diagnostiquer une opération future** : charger le JSON produit ici avec
  `att_analysis.load_reference`, puis `att_analysis.compare_operation_to_reference`
  pour positionner une opération donnée (durée vs limites SPC, défauts vs
  taux historique par pas).
- **Explorer en détail un tag** (distributions, matrice de transition,
  carte SPC, Pareto des défauts...) : `OI_ATT_v0.ipynb`.